In [ ]:
# ==========================================
# 更新说明：
# 1. 增加了自动安装 EduData 的功能 (Auto-install)。
# 2. 增加了备用下载链接 (Fallback URL)，如果不使用 EduData 也能下载。
# ==========================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ==========================================
# 1. 基础配置
# ==========================================

class Config:
    """实验超参数配置"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 100
        self.tcan_layers = 2
        self.kernel_size = 3
        self.dropout = 0.2

        # 训练参数
        self.batch_size = 64
        self.epochs = 5
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data"
        self.dataset_name = "assistment-2009-2010-skill"
        # 备用下载链接 (如果 EduData 失败)
        self.fallback_url = "https://www.cs.wpi.edu/~gendong/data/assistments/skill_builder_data.csv"

# ==========================================
# 2. 数据处理模块 (增强版: 自动安装与下载)
# ==========================================

class AssistDataset(Dataset):
    def __init__(self, q_ids, labels, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _install_edudata(self):
        """尝试自动安装 EduData"""
        print("正在尝试自动安装 EduData 库...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
            print("EduData 安装成功！")
            return True
        except Exception as e:
            print(f"自动安装失败: {e}")
            print("将尝试使用备用链接下载。")
            return False

    def download_data(self):
        """下载数据集 (优先 EduData，失败则用 URL)"""
        # 1. 检查文件是否已存在
        csv_file = self._find_csv(self.config.data_dir)
        if csv_file:
            print(f"发现已存在的数据文件: {csv_file}")
            return csv_file

        print(f"正在准备下载数据集: {self.config.dataset_name}")

        # 2. 尝试导入并使用 EduData
        use_fallback = False
        try:
            from EduData import get_data
            print("使用 EduData 下载中...")
            get_data(self.config.dataset_name, self.config.data_dir)
        except ImportError:
            # 如果没安装，尝试安装
            if self._install_edudata():
                try:
                    from EduData import get_data
                    get_data(self.config.dataset_name, self.config.data_dir)
                except Exception as e:
                    print(f"EduData 下载出错: {e}")
                    use_fallback = True
            else:
                use_fallback = True
        except Exception as e:
            print(f"EduData 运行时出错: {e}")
            use_fallback = True

        # 3. 备用方案: 直接 URL 下载
        if use_fallback:
            print(f"切换至备用方案，直接下载: {self.config.fallback_url}")
            save_path = os.path.join(self.config.data_dir, "skill_builder_data.csv")
            try:
                opener = urllib.request.build_opener()
                opener.addheaders = [('User-agent', 'Mozilla/5.0')]
                urllib.request.install_opener(opener)
                urllib.request.urlretrieve(self.config.fallback_url, save_path)
                print("下载完成！")
            except Exception as e:
                raise RuntimeError(f"所有下载方式均失败，请手动下载数据集并放入 {self.config.data_dir} 目录。错误: {e}")

        # 4. 再次查找文件
        csv_file = self._find_csv(self.config.data_dir)
        if not csv_file:
            raise FileNotFoundError("下载似乎完成了，但在目录中未找到 CSV 文件。")
        return csv_file

    def _find_csv(self, root_dir):
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith(".csv") and "skill" in file.lower():
                    return os.path.join(root, file)
        return None

    def process(self):
        file_path = self.download_data()

        print("正在读取并清洗数据 (Pandas)...")
        # 尝试不同的编码读取
        try:
            df = pd.read_csv(file_path, encoding='latin-1', low_memory=False)
        except:
            df = pd.read_csv(file_path, encoding='utf-8', low_memory=False)

        # 标准化列名
        cols = df.columns
        user_col = 'user_id'
        item_col = 'problem_id'
        skill_col = next((c for c in cols if 'skill_id' in c), None)
        if not skill_col: skill_col = next((c for c in cols if 'skill_name' in c), None)
        correct_col = 'correct'
        order_col = 'order_id'

        if not skill_col:
            raise ValueError("数据集中未找到 Skill ID 列，无法进行知识点建模。")

        # 数据清洗
        df.dropna(subset=[skill_col], inplace=True)
        df.drop_duplicates(subset=[user_col, item_col, order_col], inplace=True)

        # ID 映射
        def get_map(unique_vals):
            return {val: i+1 for i, val in enumerate(unique_vals)}

        self.config.num_students = df[user_col].nunique()

        prob_map = get_map(df[item_col].unique())
        df['item_id'] = df[item_col].map(prob_map)
        self.config.num_questions = len(prob_map) + 1

        skill_map = get_map(df[skill_col].unique())
        df['skill_id'] = df[skill_col].map(skill_map)
        self.config.num_concepts = len(skill_map) + 1

        print(f"统计: 学生 {self.config.num_students}, 题目 {len(prob_map)}, 知识点 {len(skill_map)}")

        # 构建 Q-Matrix
        print("构建 Q-Matrix...")
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['item_id', 'skill_id']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['item_id']), int(row['skill_id'])] = 1

        # 生成序列
        print("生成学生交互序列...")
        df.sort_values(by=[order_col], inplace=True)
        all_q_seqs, all_r_seqs = [], []

        # 为了演示速度，只取前1000个学生，实际实验可去掉 .head(1000)
        # df_grouped = df.groupby(user_col)
        users = df[user_col].unique()[:1000]
        df_small = df[df[user_col].isin(users)]

        for _, group in tqdm(df_small.groupby(user_col)):
            if len(group) < 5: continue
            all_q_seqs.append(group['item_id'].values)
            all_r_seqs.append(group[correct_col].values)

        return all_q_seqs, all_r_seqs, q_k_relation

# ==========================================
# 3. 核心模型定义 (HIG-TCAN-CD)
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config):
        super(HeteroGraphEmbedding, self).__init__()
        self.embed_dim = config.embed_dim
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)

    def forward(self, question_ids, q_k_relation):
        # 简单聚合：题目嵌入 = ID嵌入 + 关联知识点嵌入之和
        k_emb_weight = self.concept_emb.weight
        q_k_agg = torch.matmul(q_k_relation, k_emb_weight)
        final_q_table = self.question_emb.weight + q_k_agg
        batch_q_emb = F.embedding(question_ids, final_q_table, padding_idx=0)
        return batch_q_emb

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(self.dropout(attn), V)
        return out

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        z_perm = z.permute(0, 2, 1)
        conv_out = self.conv(z_perm)
        conv_out = conv_out[:, :, :z.size(1)].permute(0, 2, 1)
        out = self.norm(conv_out + residual)
        return self.dropout(self.relu(out))

class HIG_TCAN_CD(nn.Module):
    def __init__(self, config, q_k_relation):
        super(HIG_TCAN_CD, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])
        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, answers):
        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x)
        h = x
        for layer in self.tcan_layers:
            h = layer(h)
        return h, q_emb

# ==========================================
# 4. 实验流程
# ==========================================

def run_experiment():
    print(">>> 阶段 1: 数据准备")
    config = Config()
    processor = DataProcessor(config)

    try:
        q_seqs, r_seqs, q_k_rel = processor.process()
    except Exception as e:
        print(f"数据处理严重错误: {e}")
        return

    # 划分训练/测试集
    if len(q_seqs) == 0:
        print("错误：没有生成有效的序列数据，请检查数据源是否为空。")
        return

    train_q, test_q, train_r, test_r = train_test_split(q_seqs, r_seqs, test_size=0.2, random_state=42)

    train_dataset = AssistDataset(train_q, train_r, config.seq_len)
    test_dataset = AssistDataset(test_q, test_r, config.seq_len)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    print(f"\n>>> 阶段 2: 模型初始化 (Device: {config.device})")
    model = HIG_TCAN_CD(config, q_k_rel).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)

    print("\n>>> 阶段 3: 开始训练")
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        steps = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")
        for q_batch, r_batch, mask_batch in pbar:
            q_batch, r_batch, mask_batch = q_batch.to(config.device), r_batch.to(config.device), mask_batch.to(config.device)

            optimizer.zero_grad()
            h_seq, q_seq_emb = model(q_batch, r_batch)

            # Next Item Prediction
            h_pred = h_seq[:, :-1, :]
            q_next = q_seq_emb[:, 1:, :]
            r_target = r_batch[:, 1:]
            mask_target = mask_batch[:, 1:]

            feat = torch.cat([h_pred, q_next], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)

            loss = F.binary_cross_entropy(preds, r_target, reduction='none')
            masked_loss = (loss * mask_target).sum() / (mask_target.sum() + 1e-8)

            masked_loss.backward()
            optimizer.step()

            total_loss += masked_loss.item()
            steps += 1
            pbar.set_postfix({'Loss': f"{masked_loss.item():.4f}"})

        print(f"Epoch {epoch+1} Avg Loss: {total_loss/steps:.4f}")
        evaluate(model, test_loader, config)

def evaluate(model, loader, config):
    model.eval()
    correct = 0
    total = 0
    auc_list = []

    from sklearn.metrics import roc_auc_score

    with torch.no_grad():
        for q_batch, r_batch, mask_batch in loader:
            q_batch, r_batch, mask_batch = q_batch.to(config.device), r_batch.to(config.device), mask_batch.to(config.device)
            h_seq, q_seq_emb = model(q_batch, r_batch)

            h_pred = h_seq[:, :-1, :]
            q_next = q_seq_emb[:, 1:, :]
            r_target = r_batch[:, 1:]
            mask_target = mask_batch[:, 1:]

            feat = torch.cat([h_pred, q_next], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)

            valid_mask = mask_target.bool()
            if valid_mask.sum() == 0: continue

            valid_preds = preds[valid_mask].cpu().numpy()
            valid_targets = r_target[valid_mask].cpu().numpy()

            pred_labels = (valid_preds >= 0.5).astype(int)
            correct += (pred_labels == valid_targets).sum()
            total += len(valid_targets)

            if len(np.unique(valid_targets)) > 1:
                auc_list.append(roc_auc_score(valid_targets, valid_preds))

    acc = correct / (total + 1e-8)
    mean_auc = np.mean(auc_list) if auc_list else 0.0
    print(f"Test ACC: {acc:.4f} | Test AUC (Mean): {mean_auc:.4f}")

if __name__ == "__main__":
    run_experiment()

>>> 阶段 1: 数据准备
正在准备下载数据集: assistment-2009-2010-skill
正在尝试自动安装 EduData 库...
EduData 安装成功！


downloader, INFO http://base.ustc.edu.cn/data/ASSISTment/2009_skill_builder_data_corrected.zip is saved as data/2009_skill_builder_data_corrected.zip


downloader, INFO data/2009_skill_builder_data_corrected.zip is unzip to data/2009_skill_builder_data_corrected



正在读取并清洗数据 (Pandas)...
统计: 学生 4163, 题目 17751, 知识点 112
构建 Q-Matrix...
生成学生交互序列...


100%|██████████| 1000/1000 [00:00<00:00, 11437.28it/s]


...

Epoch 3 Avg Loss: 0.6006
Test ACC: 0.6806 | Test AUC (Mean): 0.6506


Epoch 4/5: 100%|██████████| 12/12 [00:02<00:00,  4.62it/s, Loss=0.5472]


Epoch 4 Avg Loss: 0.5911
Test ACC: 0.6817 | Test AUC (Mean): 0.6599


Epoch 5/5: 100%|██████████| 12/12 [00:02<00:00,  5.32it/s, Loss=0.5692]


Epoch 5 Avg Loss: 0.5866
Test ACC: 0.6839 | Test AUC (Mean): 0.6653
